In [ ]:
!pip install -U transformers accelerate opencv-python pandas -q
!pip install -U "typing_extensions>=4.12.2" "jinja2>=3.1.5" gradio -q
!pip install --no-deps --upgrade timm -q
!pip uninstall torchcodec -y

In [ ]:
from huggingface_hub import login

HF_TOKEN = ""  # Optional: add token if model is private
if HF_TOKEN.strip():
    login(token=HF_TOKEN)

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL_ID = "blind-assist/google-gemma-3n-2b-e3"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

print(f"Model loaded on: {model.device}")

In [ ]:
import os
import cv2
import time
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from urllib.parse import unquote

INSTRUCTION = (
    "Given the visual input from the user's forward perspective, generate exactly one short sentence to guide a visually impaired user by identifying critical obstacles or landmarks, describing their locations using clock directions relative to the user (12 o'clock is straight ahead), including relevant details such as size, material, or distance, and giving one clear action, while prioritizing immediate safety and avoiding any extra explanation."
)

def _normalize_candidate_path(candidate):
    if candidate is None:
        return None

    if isinstance(candidate, Path):
        candidate = str(candidate)

    if isinstance(candidate, os.PathLike):
        candidate = os.fspath(candidate)

    if not isinstance(candidate, str):
        return None

    text = candidate.strip()
    if not text:
        return None

    if text.startswith("file="):
        text = text.split("file=", 1)[1]
    elif "file=" in text:
        text = text.split("file=", 1)[1]

    text = unquote(text)

    if text.startswith("file://"):
        text = text.replace("file://", "", 1)

    return text

def resolve_video_path(video_value):
    if video_value is None:
        return None

    normalized = _normalize_candidate_path(video_value)
    if normalized:
        return normalized

    if isinstance(video_value, (list, tuple)):
        for item in video_value:
            resolved = resolve_video_path(item)
            if resolved:
                return resolved
        return None

    if isinstance(video_value, dict):
        preferred_keys = ["path", "video", "name", "value", "file", "url"]
        for key in preferred_keys:
            if key in video_value:
                resolved = resolve_video_path(video_value[key])
                if resolved:
                    return resolved

        for _, value in video_value.items():
            resolved = resolve_video_path(value)
            if resolved:
                return resolved
        return None

    for attr in ["path", "name", "video", "value", "file", "url"]:
        if hasattr(video_value, attr):
            try:
                resolved = resolve_video_path(getattr(video_value, attr))
                if resolved:
                    return resolved
            except Exception:
                pass

    if hasattr(video_value, "__dict__"):
        for _, value in vars(video_value).items():
            resolved = resolve_video_path(value)
            if resolved:
                return resolved

    return None

def sample_frames_every_3_seconds(video_path):
    if not video_path or not os.path.exists(video_path):
        return []

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        cap.release()
        return []

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or frame_count <= 0:
        cap.release()
        return []

    duration_sec = frame_count / fps
    timestamps = np.arange(0.0, duration_sec + 1e-9, 3.0)

    sampled = []
    for ts in timestamps:
        cap.set(cv2.CAP_PROP_POS_MSEC, float(ts * 1000.0))
        success, frame = cap.read()
        if not success:
            frame_idx = min(int(ts * fps), max(frame_count - 1, 0))
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            success, frame = cap.read()

        if not success:
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        sampled.append((float(ts), rgb))

    cap.release()
    return sampled

def frame_change_score(current_rgb, previous_rgb):
    current_gray = cv2.cvtColor(current_rgb, cv2.COLOR_RGB2GRAY)
    previous_gray = cv2.cvtColor(previous_rgb, cv2.COLOR_RGB2GRAY)

    current_small = cv2.resize(current_gray, (224, 224), interpolation=cv2.INTER_AREA)
    previous_small = cv2.resize(previous_gray, (224, 224), interpolation=cv2.INTER_AREA)

    mad = np.mean(np.abs(current_small.astype(np.float32) - previous_small.astype(np.float32)))
    return float(mad / 255.0)

def generate_navigation_guidance(frame_rgb):
    image = Image.fromarray(frame_rgb)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": INSTRUCTION},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)

    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
    )

    text = processor.decode(
        output[0][inputs["input_ids"].shape[-1] :],
        skip_special_tokens=True,
    )
    return " ".join(text.strip().split())

def analyze_video_with_threshold(video_value, change_threshold=0.08):
    video_path = resolve_video_path(video_value)
    if not video_path:
        return "No video file received. Please upload again.", []

    sampled_frames = sample_frames_every_3_seconds(video_path)
    if not sampled_frames:
        return "Could not sample frames from this video.", []

    results = []
    previous_rgb = None
    start = time.time()

    for timestamp_sec, current_rgb in sampled_frames:
        if previous_rgb is None:
            score = 1.0
            changed = True
        else:
            score = frame_change_score(current_rgb, previous_rgb)
            changed = score >= change_threshold

        if changed:
            guidance = generate_navigation_guidance(current_rgb)
        else:
            guidance = "No significant change detected; guidance skipped."

        row = {
            "time_sec": round(timestamp_sec, 2),
            "change_score": round(score, 4),
            "threshold": float(change_threshold),
            "changed": bool(changed),
            "guidance": guidance,
        }
        results.append(row)

        print(
            f"t={row['time_sec']:.2f}s | change={row['change_score']:.4f} | "
            f"changed={row['changed']} | guidance={row['guidance']}"
        )

        previous_rgb = current_rgb

    total_time = time.time() - start
    header = (
        f"Processed {len(results)} sampled timestamps (every 3s) in "
        f"{total_time:.2f}s. Threshold={change_threshold:.3f}"
    )

    log_lines = [header]
    for row in results:
        log_lines.append(
            f"t={row['time_sec']:.2f}s | score={row['change_score']:.4f} | "
            f"changed={row['changed']} | {row['guidance']}"
        )

    log_text = "\n".join(log_lines)
    table_rows = [
        [
            r["time_sec"],
            r["change_score"],
            r["threshold"],
            r["changed"],
            r["guidance"],
        ]
        for r in results
    ]

    return log_text, table_rows

In [ ]:
import gradio as gr

with gr.Blocks(title="Gemma 3N Navigation - Change Threshold") as demo:
    gr.Markdown("## 🎥 Change-Triggered Navigation Guidance (Every 3 Seconds)")

    with gr.Row():
        with gr.Column(scale=1):
            video_in = gr.Video(label="Upload Video", sources=["upload", "webcam"], format="mp4")
            threshold_in = gr.Slider(minimum=0.0, maximum=1.0, value=0.08, step=0.01, label="Change Threshold")
            run_btn = gr.Button("Analyze Video", variant="primary")

        with gr.Column(scale=1):
            log_out = gr.Textbox(label="All Guidance Logs (Per 3s Timestamp)", lines=16, interactive=False)
            table_out = gr.Dataframe(
                headers=["time_sec", "change_score", "threshold", "changed", "guidance"],
                datatype=["number", "number", "number", "bool", "str"],
                row_count=(1, "dynamic"),
                col_count=(5, "fixed"),
                label="Per-Timestamp Navigation Output",
                interactive=False,
            )

    run_btn.click(
        fn=analyze_video_with_threshold,
        inputs=[video_in, threshold_in],
        outputs=[log_out, table_out],
    )

demo.launch(share=True, debug=True)